In [13]:
#!/usr/bin/env python3
"""Kaggle Playground S5E4 – High-leverage, RAM-friendly baseline
==================================================================
* Integrates Optuna for LightGBM hyperparameter optimization.
* Finds best parameters via CV, then uses them for final prediction generation.
* **v4 Update**: Extracts numerical features from Publication_Time and Publication_Day
  *before* label encoding.
"""

import gc
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from lightgbm import LGBMRegressor
import optuna # Import Optuna

# Suppress Optuna's experimental warning if desired
optuna.logging.set_verbosity(optuna.logging.WARNING)
# Suppress LightGBM warnings about categorical features
warnings.filterwarnings("ignore", message="Using categorical_feature in Dataset.")
warnings.filterwarnings("ignore", category=UserWarning, message=".*pandas only supports SQLAlchemy connectable.*") # Suppress potential read_sql warnings if any


# --------------------------------------------------------------------------
# 1 - Memory helper
# --------------------------------------------------------------------------

def reduce_mem_usage(df: pd.DataFrame, verbose: bool = False) -> pd.DataFrame:
    start_mem = df.memory_usage(deep=True).sum() / 1024 ** 2
    for col in df.columns:
        col_type = df[col].dtype
        if pd.api.types.is_numeric_dtype(col_type):
            c_min, c_max = df[col].min(), df[col].max()
            # Skip columns that are all NaN
            if pd.isna(c_min) and pd.isna(c_max):
                continue
            if pd.api.types.is_float_dtype(col_type):
                # Check float types
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    # Check precision loss before converting to float16
                    if not np.allclose(df[col].dropna().astype(np.float16), df[col].dropna(), rtol=1e-3, atol=1e-3):
                         if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                            df[col] = df[col].astype(np.float32)
                         # else keep float64 if float32 also loses precision
                    else:
                        df[col] = df[col].astype(np.float16)

                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                # else keep float64
            else: # Integer types
                 # Check integer types only if column doesn't contain NaNs after potential coercion
                if df[col].isnull().sum() == 0:
                    if c_min >= 0: # Unsigned integers
                        if c_max < 2 ** 8:
                            df[col] = df[col].astype(np.uint8)
                        elif c_max < 2 ** 16:
                            df[col] = df[col].astype(np.uint16)
                        elif c_max < 2 ** 32:
                            df[col] = df[col].astype(np.uint32)
                        # else keep uint64 or default int64 if too large
                    else: # Signed integers
                        if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                            df[col] = df[col].astype(np.int8)
                        elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                            df[col] = df[col].astype(np.int16)
                        elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                            df[col] = df[col].astype(np.int32)
                        # else keep int64

    end_mem = df.memory_usage(deep=True).sum() / 1024 ** 2
    if verbose:
        print(f"[reduce_mem] {start_mem:0.2f}→{end_mem:0.2f} MB | −{100*(start_mem-end_mem)/start_mem:0.1f}%")
    return df

# --------------------------------------------------------------------------
# 2 - Feature engineering helpers
# --------------------------------------------------------------------------

def add_freq_and_strlen(train: pd.DataFrame, test: pd.DataFrame):
    obj_cols = train.select_dtypes(include="object").columns
    print(f"[FE] freq & strlen for {len(obj_cols)} object cols...")
    for col in obj_cols:
        # Calculate frequency on training data only to prevent leakage
        freq = train[col].value_counts(dropna=False)
        train[col + "_freq"] = train[col].map(freq).astype(np.uint32)
        # Map test data using training frequencies, fill missing with 0
        test[col + "_freq"] = test[col].map(freq).fillna(0).astype(np.uint32)

        # Calculate string length
        train[col + "_str_len"] = train[col].astype(str).str.len().astype(np.uint16)
        test[col + "_str_len"] = test[col].astype(str).str.len().astype(np.uint16)
    return train, test

def extract_datetime_features(train: pd.DataFrame, test: pd.DataFrame):
    """Extracts numerical features from Publication_Time and Publication_Day."""
    print("[FE] Extracting datetime features...")
    added_features = []

    # --- Process Publication_Time ---
    time_col = 'Publication_Time'
    if time_col in train.columns and time_col in test.columns:
        for df in [train, test]:
            # Convert to datetime, coercing errors to NaT
            time_dt = pd.to_datetime(df[time_col], format='%H:%M:%S', errors='coerce')

            # Extract features, handle NaT by keeping them as NaN initially
            df['time_hour'] = time_dt.dt.hour.astype(pd.Int8Dtype()) # Use nullable integer type
            df['time_minute'] = time_dt.dt.minute.astype(pd.Int8Dtype())
            df['time_total_seconds'] = (time_dt.dt.hour * 3600 +
                                        time_dt.dt.minute * 60 +
                                        time_dt.dt.second).astype(pd.Int32Dtype())
        added_features.extend(['time_hour', 'time_minute', 'time_total_seconds'])
        print(f"[FE] Added time features: time_hour, time_minute, time_total_seconds")

    # --- Process Publication_Day ---
    day_col = 'Publication_Day'
    if day_col in train.columns and day_col in test.columns:
         # Define mapping based on common English day names
        day_map = {
            'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3,
            'Friday': 4, 'Saturday': 5, 'Sunday': 6
        }
        weekend_days = {5, 6} # Saturday, Sunday

        for df in [train, test]:
            df['day_of_week'] = df[day_col].map(day_map).astype(pd.Int8Dtype()) # Keep NaN for unknown days
            df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if pd.notna(x) and x in weekend_days else (0 if pd.notna(x) else pd.NA) ).astype(pd.Int8Dtype())

        added_features.extend(['day_of_week', 'is_weekend'])
        print(f"[FE] Added day features: day_of_week, is_weekend")

    print(f"[FE] Datetime extraction complete. Added {len(added_features)} features.")
    # Note: NaNs introduced here will be handled by the SimpleImputer later
    return train, test


def selective_log1p(train: pd.DataFrame, test: pd.DataFrame, skew_thresh: float = 1.2):
    """Create log1p features for *positive* numeric columns with high skew.
    Cast to float64 before skew calculation to avoid overflow warnings.
    Handles potential pd.NA values."""
    num_cols = train.select_dtypes(include=np.number).columns
    added = 0
    print(f"[FE] log1p transformation (skew > {skew_thresh})...")
    for col in num_cols:
        # Convert to numeric, coercing errors. Handles existing numeric types too.
        numeric_col_train = pd.to_numeric(train[col], errors='coerce')

        # Check if the column is entirely NA after coercion
        if numeric_col_train.isnull().all():
            # print(f"Skipping log1p for {col} as it contains only NA values.")
            continue

        # Check if *any* non-NA value is non-positive (<= 0)
        # This correctly handles pd.NA by ignoring it in the check
        if (numeric_col_train.dropna() <= 0).any():
            # print(f"Skipping log1p for {col} as it contains non-positive values.")
            continue

        # Proceed only if all non-NA values are positive
        col_float = numeric_col_train.astype(np.float64) # Cast for robust skew calculation
        skew_val = col_float.skew()

        # Check if skew calculation is valid and exceeds threshold
        if pd.notna(skew_val) and abs(skew_val) > skew_thresh:
            try:
                 # Apply log1p transformation - log1p handles NaNs correctly
                train[col + "_log1p"] = np.log1p(col_float).astype(np.float32)
                # Ensure test transformation handles potential different dtypes or NaNs safely
                numeric_col_test = pd.to_numeric(test[col], errors='coerce').astype(np.float64)
                test[col + "_log1p"] = np.log1p(numeric_col_test).astype(np.float32)
                added += 1
                # print(f"Applied log1p to {col} (skew={skew_val:.2f})")
            except Exception as e:
                 print(f"Warning: Could not apply log1p to {col}. Error: {e}")

    print(f"[FE] log1p added on {added} numeric cols")
    return train, test


def label_encode(train_df: pd.DataFrame, test_df: pd.DataFrame):
    # Identify object columns *after* potential datetime feature extraction
    obj_cols = train_df.select_dtypes(include="object").columns
    cat_cols = [] # Keep track of encoded columns for later use (e.g., LightGBM)
    print(f"[encode] Label encoding {len(obj_cols)} object cols...")
    for col in obj_cols:
        le = LabelEncoder()
        # Combine train and test for a complete mapping of categories
        # Handle potential NaNs by converting to string first
        full = pd.concat([train_df[col].astype(str), test_df[col].astype(str)], axis=0)
        le.fit(full)

        # Transform train and test, ensuring consistent data types
        train_df[col] = le.transform(train_df[col].astype(str)).astype(np.uint32)
        test_df[col] = le.transform(test_df[col].astype(str)).astype(np.uint32)
        cat_cols.append(col) # Add to list of categorical features

    print(f"[encode] Label-encoded {len(cat_cols)} cols: {cat_cols}")
    return train_df, test_df, cat_cols


def add_target_mean_encoding(X: pd.DataFrame, X_test: pd.DataFrame, y: pd.Series, cat_cols, n_splits: int = 5):
    print(f"[TME] Target-mean encoding {len(cat_cols)} categorical cols...")
    global_mean = y.mean() # Calculate global mean for filling missing mappings
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42) # Use KFold for robust encoding

    for col in cat_cols:
        if col not in X.columns: # Skip if column was somehow dropped
             print(f"Warning: Column {col} not found for TME, skipping.")
             continue

        oof = np.zeros(len(X), dtype=np.float32) # Out-of-fold predictions for train set
        test_enc = np.zeros(len(X_test), dtype=np.float32) # Test set encoding averaged over folds

        # Iterate through folds to calculate target means without leakage
        for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y)):
            X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
            y_tr = y.iloc[tr_idx]

            # Calculate means on the training part of the fold
            # Ensure grouping column is treated correctly (it's numeric after LE)
            means = y_tr.groupby(X_tr[col]).mean()

            # Apply means to the validation part of the fold
            oof[val_idx] = X_val[col].map(means).fillna(global_mean).values

            # Apply means to the test set and accumulate (average later)
            test_enc += X_test[col].map(means).fillna(global_mean).values

        # Assign the OOF encodings to the training set
        X[col + "_te"] = oof.astype(np.float32)
        # Assign the averaged encodings to the test set
        X_test[col + "_te"] = (test_enc / n_splits).astype(np.float32)

    print(f"[TME] Added target mean encoding for {len(cat_cols)} cols.")
    return X, X_test


# --------------------------------------------------------------------------
# 3 - Load data
# --------------------------------------------------------------------------

BASE_DIR = Path("/kaggle/input/playground-series-s5e4")
# Use local paths if Kaggle input doesn't exist (for local testing)
train_path = BASE_DIR / "train.csv" if BASE_DIR.exists() else Path("train.csv")
test_path = BASE_DIR / "test.csv" if BASE_DIR.exists() else Path("test.csv")
sub_path = BASE_DIR / "sample_submission.csv" if BASE_DIR.exists() else Path("sample_submission.csv")

print("[main] Load data...")
train = pd.read_csv(train_path, low_memory=False)
test = pd.read_csv(test_path, low_memory=False)
print("[main] Shapes – train", train.shape, "test", test.shape)

# Original IDs might be useful but remove if not needed for modeling
train_ids = train['id']
test_ids = test['id']
train = train.drop(columns=['id'])
test = test.drop(columns=['id'])


train = reduce_mem_usage(train, verbose=True)
test = reduce_mem_usage(test, verbose=True)

target = "Listening_Time_minutes"
assert target in train.columns, f"Target column '{target}' not found in training data."

# --------------------------------------------------------------------------
# 4 - Feature Engineering
# --------------------------------------------------------------------------

# --- Apply FE steps in logical order ---
# 1. Freq/Strlen on original object columns
train, test = add_freq_and_strlen(train, test)

# 2. Extract numerical datetime features *before* label encoding originals
train, test = extract_datetime_features(train, test)

# 3. Label Encode remaining object columns (incl. original Time/Day)
train, test, cat_cols_encoded = label_encode(train, test) # Get list of *encoded* categorical columns

# Separate target variable
y = train[target].astype(np.float32)
X = train.drop(columns=[target])

# Align columns *before* TME and log transform
X_cols = list(X.columns)
test = test[X_cols] # Reorder/subset test columns to match X

# 4. Target Mean Encoding on *label-encoded* categorical columns
#    'cat_cols_encoded' now holds the names of the columns that were label encoded
X, test = add_target_mean_encoding(X, test, y, cat_cols_encoded, n_splits=5)

# 5. Log Transform highly skewed *positive* numeric columns (check all numeric cols)
X, test = selective_log1p(X, test, skew_thresh=1.2)

# Reduce memory after adding features
X = reduce_mem_usage(X, verbose=True)
test = reduce_mem_usage(test, verbose=True)

# 6. Impute missing values *after* all feature engineering
print("[main] Imputing missing values with median...")
# Select columns that are *actually* numeric (standard or nullable)
numeric_cols_final = X.select_dtypes(include=np.number).columns
print(f"[main] Columns selected for imputation ({len(numeric_cols_final)}): {numeric_cols_final.tolist()}")

if not numeric_cols_final.empty: # Proceed only if there are numeric columns
    imp = SimpleImputer(strategy="median")

    # Fit on Training data only - Convert to standard float64 for imputer compatibility
    try:
        X_numeric_float64 = X[numeric_cols_final].astype(np.float64)
        imp.fit(X_numeric_float64)

        # --- Transform Train ---
        # 1. Transform (get NumPy array)
        imputed_data_train = imp.transform(X_numeric_float64) # Use already converted array
        # 2. Convert back to DataFrame with original index and columns
        imputed_df_train = pd.DataFrame(imputed_data_train, index=X.index, columns=numeric_cols_final)
        # 3. Assign the DataFrame back (safer assignment)
        X[numeric_cols_final] = imputed_df_train
        print("[main] Imputation applied to Training data.")

        # --- Transform Test ---
        # 1. Convert Test slice to float64
        test_numeric_float64 = test[numeric_cols_final].astype(np.float64)
         # 2. Transform (get NumPy array)
        imputed_data_test = imp.transform(test_numeric_float64)
        # 3. Convert back to DataFrame with original index and columns
        imputed_df_test = pd.DataFrame(imputed_data_test, index=test.index, columns=numeric_cols_final)
        # 4. Assign the DataFrame back
        test[numeric_cols_final] = imputed_df_test
        print("[main] Imputation applied to Test data.")

    except Exception as e:
        print(f"Error during imputation: {e}")
        print("Skipping imputation step due to error.")

else:
    print("[main] No numeric columns found for imputation. Skipping step.")


# Convert back after imputation and apply final memory reduction
X = reduce_mem_usage(X, verbose=True)
test = reduce_mem_usage(test, verbose=True)

# Identify categorical features for LightGBM (the ones that were originally object and got label encoded)
# These are now integer types but should be treated as categorical by the model
lgbm_cat_features = cat_cols_encoded # Use the list returned by label_encode
print(f"[main] Identified {len(lgbm_cat_features)} features for LightGBM categorical handling: {lgbm_cat_features}")


# Final check for column alignment
assert list(X.columns) == list(test.columns), "Train and test columns mismatch after FE."
print("[main] Feature Engineering complete. Final shapes: X", X.shape, "test", test.shape)
print(f"Features: {list(X.columns)}")


# Clean up memory before HPO/Training
del train # Original train DF no longer needed
gc.collect()

# --------------------------------------------------------------------------
# 5 - Optuna Hyperparameter Optimization
# --------------------------------------------------------------------------

# def objective(trial, X_train, y_train, cat_features_list, n_splits=5):
#     """Objective function for Optuna study."""

#     # Define hyperparameter search space
#     params = {
#         "objective": "rmse",
#         "metric": "rmse",
#         "boosting_type": "gbdt",
#         "n_estimators": 4000, # Keep high, rely on early stopping
#         "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
#         "num_leaves": trial.suggest_int("num_leaves", 31, 511), # Wider range
#         "max_depth": trial.suggest_int("max_depth", -1, 15), # Allow unlimited or limited depth
#         "subsample": trial.suggest_float("subsample", 0.5, 1.0), # Bagging fraction
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0), # Feature fraction
#         "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
#         "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True), # L1 regularization
#         "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True), # L2 regularization
#         "n_jobs": -1,
#         "random_state": 42, # Fixed seed for consistent HPO runs
#         "verbose": -1, # Suppress LightGBM verbosity during HPO
#     }

#     kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
#     fold_rmses = []

#     # --- Ensure categorical features are correctly handled by LightGBM ---
#     # LightGBM expects a list of column *names* or 'auto'.
#     # Ensure all specified cat_features_list exist in X_train.columns
#     valid_cat_features = [col for col in cat_features_list if col in X_train.columns]
#     if not valid_cat_features:
#         categorical_feature_param = 'auto' # Let LightGBM detect if list is empty
#         print("Warning: No valid categorical features found from list for HPO fold. Using 'auto'.")
#     else:
#         categorical_feature_param = valid_cat_features


#     for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train, y_train)):
#         X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
#         y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

#         model = LGBMRegressor(**params)
#         model.fit(
#             X_tr, y_tr,
#             eval_set=[(X_val, y_val)],
#             eval_metric="rmse",
#             categorical_feature=categorical_feature_param, # Pass the list of names
#             callbacks=[lgb.early_stopping(100, first_metric_only=True, verbose=False)] # Increased patience
#         )

#         preds_val = model.predict(X_val)
#         rmse = mean_squared_error(y_val, preds_val, squared=False)
#         fold_rmses.append(rmse)
#         # Optional: Pruning based on intermediate fold performance (can speed up HPO)
#         # trial.report(rmse, fold)
#         # if trial.should_prune():
#         #     raise optuna.exceptions.TrialPruned()


#     avg_rmse = np.mean(fold_rmses)
#     return avg_rmse # Optuna minimizes this value


# print("[HPO] Starting Optuna hyperparameter optimization...")
# # You might need to install optuna: pip install optuna
# study = optuna.create_study(
#     direction="minimize",
#     study_name="lgbm_podcast_listening_time_v4",
#     #sampler=optuna.samplers.TPESampler(seed=42) # Default TPE sampler is good
#     )

# N_TRIALS = 2 # Set N_TRIALS to a desired number (e.g., 50-100) for a proper search. 1-2 for quick testing.
# print(f"[HPO] Running Optuna study for {N_TRIALS} trials...")

# # Run optimization - Increase n_trials for better results, decrease for speed
# # Pass X, y, and the *names* of the label-encoded columns
# study.optimize(lambda trial: objective(trial, X, y, lgbm_cat_features, n_splits=5), n_trials=N_TRIALS)

# print(f"[HPO] Optuna study finished. Number of trials: {len(study.trials)}")
# print(f"[HPO] Best trial's CV RMSE: {study.best_value:.5f}")
print("[HPO] Best hyperparameters found:")
# best_params = study.best_params

best_params = {'learning_rate': 0.07200157408568503, 
               'num_leaves': 467, 
               'max_depth': 12, 
               'subsample': 0.7674040101752828, 
               'colsample_bytree': 0.7427940378489676, 
               'min_child_samples': 72, 
               'reg_alpha': 0.003620326587094506, 
               'reg_lambda': 4.30681230814926}

print(best_params)

# Optionally hardcode params if needed (e.g., from a previous run)
# best_params = {'learning_rate': 0.0..., 'num_leaves': ..., ...}
# print("[HPO] Using pre-defined best hyperparameters.")
# print(best_params)


# Clean up memory before final training
gc.collect()

# --------------------------------------------------------------------------
# 5.5 - Feature Selection based on Importance (from previous run)
# --------------------------------------------------------------------------
print("\n[main] Performing feature selection based on pre-calculated importance...")

# --- Define features to drop based on the provided importance list ---
# Features with mean importance = 0.0 in the provided list:
zero_importance_features = [
    'time_minute',
    'time_total_seconds',
    'time_hour',
    'Episode_Title_str_len_log1p',
    'Episode_Sentiment_str_len',
    'Episode_Title_str_len',
    'is_weekend',
    'Genre_str_len',
    'Episode_Sentiment_freq',
    'Publication_Time_str_len',
    'Genre_freq',
    'Publication_Day_str_len',
    'Publication_Time',
    'Podcast_Name_str_len',
]

# Optional: Add features with very low importance if desired
# low_importance_features = [
#     'Episode_Title_str_len_log1p',
#     'Episode_Sentiment_str_len',
#     # ... add others based on a threshold (e.g., mean < 100)
# ]
# features_to_drop = zero_importance_features + low_importance_features

features_to_drop = zero_importance_features # Start by dropping only zero-importance features

# --- Ensure features exist before attempting to drop ---
features_present_in_X = [f for f in features_to_drop if f in X.columns]
features_present_in_test = [f for f in features_to_drop if f in test.columns]

if not features_present_in_X:
    print("[main] No features to drop found in the current DataFrame X.")
else:
    print(f"[main] Dropping {len(features_present_in_X)} features: {features_present_in_X}")
    X = X.drop(columns=features_present_in_X)
    # Ensure test columns align if features were dropped from X
    if features_present_in_test == features_present_in_X:
         test = test.drop(columns=features_present_in_test)
    else:
        # This case should ideally not happen if X and test were aligned before
        print("Warning: Mismatch in features to drop between X and test. Re-aligning test.")
        # Keep only columns that are still in X
        test = test[X.columns] # Realign test based on remaining columns in X


    # --- Update the list of categorical features for LightGBM ---
    original_cat_count = len(lgbm_cat_features)
    lgbm_cat_features = [col for col in lgbm_cat_features if col not in features_present_in_X]
    print(f"[main] Updated lgbm_cat_features list (removed {original_cat_count - len(lgbm_cat_features)} dropped features).")
    print(f"[main] Remaining {len(lgbm_cat_features)} features for LightGBM categorical handling: {lgbm_cat_features}")


    # --- Final check and memory reduction ---
    assert list(X.columns) == list(test.columns), "Train and test columns mismatch after feature selection."
    print(f"[main] Feature selection complete. New shapes: X {X.shape}, test {test.shape}")
    X = reduce_mem_usage(X, verbose=True)
    test = reduce_mem_usage(test, verbose=True)
    gc.collect()


# --------------------------------------------------------------------------
# 6 - Final LightGBM Training with Best Parameters
# --------------------------------------------------------------------------

# Use the best parameters found by Optuna, but keep essential ones
final_lgbm_params = {
    "objective": "rmse",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "n_estimators": 6000, # Use a high value, rely on early stopping per fold
    "n_jobs": -1,
    "verbose": -1,
    # ** Add best params found by Optuna **
    **best_params
}


kf = KFold(n_splits=5, shuffle=True, random_state=42) # Use consistent folds if possible
print(f"[main] Training final 5-fold LightGBM with best params: {final_lgbm_params}")

oof_final = np.zeros(len(X), dtype=np.float32)
ptest_final = np.zeros(len(test), dtype=np.float32)
final_fold_scores = []
feature_importances = pd.DataFrame(index=X.columns)

# Ensure categorical features are correctly handled in final training
final_valid_cat_features = [col for col in lgbm_cat_features if col in X.columns]
if not final_valid_cat_features:
    final_categorical_feature_param = 'auto'
    print("Warning: No valid categorical features found from list for final training. Using 'auto'.")
else:
    final_categorical_feature_param = final_valid_cat_features


for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y), 1):
    print(f"[fold {fold}] train {len(tr_idx)} • val {len(val_idx)}")
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    # Add random state specific to fold for final model diversity if desired
    fold_params = final_lgbm_params.copy()
    fold_params['random_state'] = fold * 123 # Different seed per fold

    model = LGBMRegressor(**fold_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="rmse",
        categorical_feature=final_categorical_feature_param, # Pass category names
        callbacks=[lgb.early_stopping(100, first_metric_only=True, verbose=100)] # More patience, less verbose
    )

    best_it = model.best_iteration_ if model.best_iteration_ else model.n_estimators_
    preds_val = model.predict(X_val, num_iteration=best_it)
    oof_final[val_idx] = preds_val
    ptest_final += model.predict(test, num_iteration=best_it) / kf.n_splits
    rmse = mean_squared_error(y_val, preds_val, squared=False)
    final_fold_scores.append(rmse)
    feature_importances[f'fold_{fold}'] = model.feature_importances_
    print(f"[fold {fold}] best_iter={best_it}, Val. RMSE={rmse:0.5f}")
    del model, X_tr, X_val, y_tr, y_val
    gc.collect()

cv_rmse_final = mean_squared_error(y, oof_final, squared=False)
print(f"\n[main] Final OOF CV RMSE with best params: {cv_rmse_final:0.5f}")
print(f"[main] Per-fold RMSEs: {[f'{s:.5f}' for s in final_fold_scores]}")
print(f"[main] Average Fold RMSE: {np.mean(final_fold_scores):0.5f} +/- {np.std(final_fold_scores):0.5f}")

# Display feature importances
feature_importances['mean'] = feature_importances.mean(axis=1)
feature_importances['std'] = feature_importances.std(axis=1)
print("\n[main] Top 20 Feature Importances (mean over folds):")
print(feature_importances.sort_values('mean', ascending=False).head(20))


# --------------------------------------------------------------------------
# 7 - Submission
# --------------------------------------------------------------------------

print("\n[main] Creating submission file...")
sub = pd.read_csv(sub_path) # Load sample submission
sub[target] = np.maximum(0, ptest_final) # Ensure predictions are non-negative
# Ensure 'id' column exists in sub before assignment (it should from sample_submission)
if 'id' in sub.columns:
     sub['id'] = test_ids # Add original IDs back
else:
     print("Warning: 'id' column not found in sample submission. Creating it.")
     sub['id'] = test_ids

output_filename = "submission.csv"
sub[['id', target]].to_csv(output_filename, index=False) # Ensure correct columns and order
print(f"[main] ✓ {output_filename} saved. ({sub.shape})")
print(sub[['id', target]].head())
print("\n--- Script Execution Finished ---")

[main] Load data...
[main] Shapes – train (750000, 12) test (250000, 11)
[reduce_mem] 310.30→288.84 MB | −6.9%
[reduce_mem] 101.52→96.28 MB | −5.2%
[FE] freq & strlen for 6 object cols...
[FE] Extracting datetime features...
[FE] Added time features: time_hour, time_minute, time_total_seconds
[FE] Added day features: day_of_week, is_weekend
[FE] Datetime extraction complete. Added 5 features.
[encode] Label encoding 6 object cols...
[encode] Label-encoded 6 cols: ['Podcast_Name', 'Episode_Title', 'Genre', 'Publication_Day', 'Publication_Time', 'Episode_Sentiment']
[TME] Target-mean encoding 6 categorical cols...
[TME] Added target mean encoding for 6 cols.
[FE] log1p transformation (skew > 1.2)...
[FE] log1p added on 1 numeric cols
[reduce_mem] 77.96→46.49 MB | −40.4%
[reduce_mem] 26.46→15.97 MB | −39.6%
[main] Imputing missing values with median...
[main] Columns selected for imputation (34): ['Podcast_Name', 'Episode_Title', 'Episode_Length_minutes', 'Genre', 'Host_Popularity_percent